# Transferable NISAR and SNOTEL workflow

This notebook is a reviewer and user template. Replace the Colorado paths with a different NISAR GUNW, analysis box, and SNOTEL station table to test whether the workflow transfers to a new study area.

SnowIn handles product reading, phase normalization, incidence geometry, reference-offset application, dSWE, support, and provenance. Station/date selection and the construction of expected phase remain caller-owned so that a study's SNOTEL policy is visible rather than hidden in the package.

## 1. Configure your local inputs

Set these environment variables before launching Jupyter, or enter the required GUNW path when prompted:

- `SNOWIN_GUNW`: one local NISAR GUNW product.
- `SNOWIN_NISAR_DEM`: optional local NISAR-modified Copernicus DEM; otherwise SnowIn uses its normal cache/download path.
- `SNOWIN_ANALYSIS_VECTOR`: optional GeoPackage, Shapefile, or GeoJSON analysis region.
- `SNOWIN_ANALYSIS_LAYER`: optional layer name for a multi-layer GeoPackage.
- `SNOWIN_SNOTEL_TABLE`: optional CSV with station rows.
- `SNOWIN_REFERENCE_OFFSET_RAD`: optional manual offset when station reference inputs are not ready.

For station-based referencing, the CSV should contain `station_id`, `latitude`, `longitude`, `expected_phase_rad`, and `coherence_weight`. An optional `observed_phase_rad` column can be used when phase was sampled externally; otherwise this notebook samples the canonical SnowIn phase at each station location. Optional `eligible` and `exclusion_reason` columns are preserved as explicit support rules.

In [ ]:
import csv
import math
import os
from html import escape
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import HTML, display
from pyproj import Transformer

from snowin import compute_dswe, reference_phase
from snowin.io import add_gunw_incidence, open_gunw
from snowin.spatial import rasterize_vector_mask

In [ ]:
def configured_path(name, *, prompt=False):
    value = os.environ.get(name)
    if not value and prompt:
        value = input(f'{name} path: ').strip()
    if not value:
        return None
    path = Path(value).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f'{name} does not exist: {path}')
    return path

GUNW_PATH = configured_path('SNOWIN_GUNW', prompt=True)
if GUNW_PATH is None:
    raise RuntimeError('A local NISAR GUNW is required for this notebook.')
DEM_PATH = configured_path('SNOWIN_NISAR_DEM')
VECTOR_PATH = configured_path('SNOWIN_ANALYSIS_VECTOR')
STATION_TABLE = configured_path('SNOWIN_SNOTEL_TABLE')
VECTOR_LAYER = os.environ.get('SNOWIN_ANALYSIS_LAYER')
REFERENCE_OFFSET = os.environ.get('SNOWIN_REFERENCE_OFFSET_RAD')
REFERENCE_OFFSET = float(REFERENCE_OFFSET) if REFERENCE_OFFSET else None

try:
    import dask.array  # noqa: F401
    CHUNKS = 'auto'
except ImportError:
    CHUNKS = None

print('GUNW:', GUNW_PATH)
print('NISAR DEM:', DEM_PATH or 'SnowIn cache/download path')
print('Analysis vector:', VECTOR_PATH or 'not configured')
print('SNOTEL table:', STATION_TABLE or 'not configured')
print('Chunks:', CHUNKS or 'eager')

## 2. Open and inspect the GUNW before geometry

This is the fast inspection step. It should expose a true xarray Dataset, normalized phase, coordinates, acquisition metadata, wavelength, and source provenance before the slower DEM/LOS calculation runs.

In [ ]:
pair = open_gunw(GUNW_PATH, chunks=CHUNKS, progress=True)
display(pair)
display(HTML('<h4>Selected provenance</h4><pre>' + escape(str({k: pair.attrs.get(k) for k in ["phase_difference_definition", "source_phase_difference_definition", "phase_transform", "wavelength_m", "wavelength_source", "reference_time", "secondary_time"]})) + '</pre>'))

## 3. Add incidence explicitly

The GUNW path is supplied again deliberately. `add_gunw_incidence()` appends local `incidence_angle` and the boolean `geometry_valid` mask to the same Dataset, and records the DEM, datum, LOS, and geometry provenance. Unsupported and back-facing terrain remains missing.

In [ ]:
incidence_kwargs = {'dem_source': 'nisar_cop30', 'progress': True}
if DEM_PATH is not None:
    incidence_kwargs['nisar_cop30_dem'] = DEM_PATH

add_gunw_incidence(pair, GUNW_PATH, **incidence_kwargs)
display(pair)
display(HTML('<h4>Geometry provenance</h4><pre>' + escape(str({k: pair.attrs.get(k) for k in ["incidence_angle_source", "incidence_angle_reference", "dem_source", "dem_product", "dem_height_reference", "gunw_height_reference", "vertical_datum_status", "geometry_execution"]})) + '</pre>'))
print(f'Geometry-supported fraction: {float(pair.geometry_valid.mean()):.6f}')

## 4. Add an explicit analysis-region mask

The region mask is applied after opening the product because the GUNW supplies the target grid and CRS. It restricts plots, support summaries, and metrics; it does not alter raw phase, phase normalization, or station selection.

In [ ]:
if VECTOR_PATH is not None:
    mask_kwargs = {'target': pair, 'progress': True}
    if VECTOR_LAYER:
        mask_kwargs['layer'] = VECTOR_LAYER
    pair['analysis_region'] = rasterize_vector_mask(VECTOR_PATH, **mask_kwargs)
    print(f'Analysis-region fraction: {float(pair.analysis_region.mean()):.6f}')
    display(pair[['analysis_region']])
else:
    print('No analysis vector configured; metrics below use the full GUNW grid.')

## 5. Reference phase with caller-owned SNOTEL inputs

SnowIn does not decide which SNOTEL station dates, snow model, station allowlist, or basin policy to use. This cell demonstrates the boundary: the caller supplies station IDs, coordinates, observed canonical phase, expected canonical phase, weights, and exclusions.

If `observed_phase_rad` is not present, the notebook samples `pair.phase` at the station coordinates. The expected phase must already be computed by the caller from the SNOTEL snow record and the selected physical model. Both observed and expected phase must use SnowIn's canonical `secondary_minus_reference` convention.

In [ ]:
pair_referenced = None
station_summary = None

if STATION_TABLE is not None:
    with STATION_TABLE.open(newline='') as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise ValueError('SNOTEL table has no data rows.')
    required = {'station_id', 'latitude', 'longitude', 'expected_phase_rad', 'coherence_weight'}
    missing = required - set(rows[0])
    if missing:
        raise ValueError(f'SNOTEL table is missing required columns: {sorted(missing)}')

    station_ids = np.asarray([row['station_id'] for row in rows], dtype=str)
    latitudes = np.asarray([float(row['latitude']) for row in rows])
    longitudes = np.asarray([float(row['longitude']) for row in rows])
    expected = xr.DataArray(
        np.asarray([float(row['expected_phase_rad']) for row in rows]),
        dims=('reference_contributor',),
        attrs={'units': 'rad'},
        name='expected_phase',
    )
    weights = xr.DataArray(
        np.asarray([float(row['coherence_weight']) for row in rows]),
        dims=('reference_contributor',),
        attrs={'units': '1'},
        name='coherence_weight',
    )
    epsg = int(pair.spatial_ref.attrs['epsg_code'])
    transformer = Transformer.from_crs(4326, epsg, always_xy=True)
    station_x, station_y = transformer.transform(longitudes, latitudes)
    sampled = pair['phase'].sel(
        x=xr.DataArray(station_x, dims='reference_contributor'),
        y=xr.DataArray(station_y, dims='reference_contributor'),
        method='nearest',
    ).load().rename('observed_phase')
    sampled.attrs = {'units': 'rad', 'sampling': 'nearest GUNW pixel'}
    if 'observed_phase_rad' in rows[0]:
        observed = xr.DataArray(
            np.asarray([float(row['observed_phase_rad']) for row in rows]),
            dims=('reference_contributor',), attrs={'units': 'rad'}, name='observed_phase',
        )
        print('Using observed_phase_rad supplied by the station table.')
    else:
        observed = sampled
        print('Using observed phase sampled from pair.phase at station locations.')
    eligible = xr.DataArray(
        np.asarray([row.get('eligible', 'true').lower() not in {'false', '0', 'no'} for row in rows]),
        dims=('reference_contributor',), name='eligible',
    )
    reasons = xr.DataArray(
        np.asarray([row.get('exclusion_reason') or 'eligible' for row in rows], dtype=str),
        dims=('reference_contributor',), name='exclusion_reason',
    )
    reasons = reasons.where(eligible, 'caller_excluded')
    pair_referenced = reference_phase(
        pair, observed, expected, weights,
        contributor_id=xr.DataArray(station_ids, dims='reference_contributor'),
        exclusion_reason=reasons,
        method='coherence_weighted',
    )
    station_summary = pair_referenced[[
        'reference_contributor_id', 'reference_observed_phase',
        'reference_expected_phase', 'reference_weight',
        'reference_exclusion_reason',
    ]]
    display(station_summary)
    print('Reference offset [rad]:', pair_referenced.attrs.get('phase_reference_offset_rad'))
elif REFERENCE_OFFSET is not None:
    pair_referenced = reference_phase(pair, method='manual_offset', offset_rad=REFERENCE_OFFSET)
    print('Applied explicit manual reference offset [rad]:', REFERENCE_OFFSET)
else:
    print('No station table or manual offset configured; set one before computing dSWE.')

## 6. Compute dSWE and inspect support

The incidence domain is made explicit before calling the generic kernel. Unsupported incidence values remain NaN; they are not converted to zero.

In [ ]:
if pair_referenced is not None:
    valid_incidence = pair_referenced['incidence_angle'].where(pair_referenced['geometry_valid'])
    dswe = compute_dswe(
        pair_referenced['phase_referenced'],
        valid_incidence,
        wavelength_m=float(pair_referenced.attrs['wavelength_m']),
    )
    pair_referenced['dswe'] = dswe
    pair_referenced['geometry_support'] = valid_incidence.notnull()
    pair_referenced['retrieval_support'] = pair_referenced['dswe'].notnull()
    if 'analysis_region' in pair_referenced:
        pair_referenced['final_support'] = (
            pair_referenced['analysis_region']
            & pair_referenced['geometry_support']
            & pair_referenced['retrieval_support']
        )
    display(pair_referenced[['phase', 'phase_referenced', 'incidence_angle', 'dswe', 'geometry_support', 'retrieval_support']])
    support_name = 'final_support' if 'final_support' in pair_referenced else 'retrieval_support'
    print(f'{support_name} fraction:', float(pair_referenced[support_name].mean()))
else:
    print('dSWE skipped until a station table or manual reference offset is supplied.')

In [ ]:
if pair_referenced is not None and 'dswe' in pair_referenced:
    variables = [
        ('phase_referenced', 'Referenced phase [rad]'),
        ('incidence_angle', 'Local incidence [rad]'),
        ('dswe', 'Pairwise dSWE [m]'),
    ]
    fig, axes = plt.subplots(1, len(variables), figsize=(16, 4), constrained_layout=True)
    for axis, (name, title) in zip(axes, variables):
        layer = pair_referenced[name]
        if 'analysis_region' in pair_referenced:
            layer = layer.where(pair_referenced['analysis_region'])
        layer.plot.imshow(ax=axis, robust=True, add_colorbar=True)
        axis.set_title(title)
    plt.show()
else:
    print('Plots will appear after reference and dSWE inputs are configured.')

## 7. Transferability checks

Review these checks for every new NISAR box. They are deliberately visible so reviewers can distinguish a package failure from a study-input or station-policy issue.

In [ ]:
checks = {
    'xarray_dataset': isinstance(pair, xr.Dataset),
    'canonical_phase_definition': pair.attrs.get('phase_difference_definition') == 'secondary_minus_reference',
    'phase_transform_recorded': bool(pair.attrs.get('phase_transform')),
    'wavelength_positive': float(pair.attrs.get('wavelength_m', np.nan)) > 0.0,
    'wavelength_source_recorded': bool(pair.attrs.get('wavelength_source')),
    'incidence_present': 'incidence_angle' in pair,
    'geometry_valid_present': 'geometry_valid' in pair,
    'incidence_units_radians': pair.get('incidence_angle', xr.DataArray()).attrs.get('units') == 'rad',
    'incidence_reference_recorded': pair.get('incidence_angle', xr.DataArray()).attrs.get('incidence_angle_reference') in {'local', 'ellipsoid'},
    'dem_source_recorded': bool(pair.attrs.get('dem_source')),
}
if VECTOR_PATH is not None:
    checks['analysis_region_present'] = 'analysis_region' in pair
if pair_referenced is not None:
    checks['reference_status_recorded'] = bool(pair_referenced.attrs.get('phase_reference_status'))
    checks['dswe_present'] = 'dswe' in pair_referenced
for name, passed in checks.items():
    print(f'{"PASS" if passed else "REVIEW":>6}  {name}')

print('Phase definition:', pair.attrs.get('phase_difference_definition'))
print('Wavelength [m]:', pair.attrs.get('wavelength_m'))
print('Wavelength source:', pair.attrs.get('wavelength_source'))
print('DEM source:', pair.attrs.get('dem_source'))
print('Incidence source:', pair.attrs.get('incidence_angle_source'))

## Reviewer record

Before reporting that a new box transferred successfully, record:

1. GUNW granule, polarization, frequency, and box/vector path.
2. DEM path/source and wavelength source.
3. Phase definition and transform provenance.
4. SNOTEL station IDs, date-matching rule, expected-phase construction, weights, and exclusions.
5. Geometry and retrieval support fractions.
6. Any errors, warnings, or usability issues.

For the Colorado acceptance check, Notebook 03 should reproduce the documented basin-matched metrics before any differences are interpreted as a SnowIn defect.